In [1]:
"""
HP Landslide Model — Final Training Script (v7)
================================================
Works with hp_landslide_dataset_v7_final.csv
New: includes aspect feature, bigger dataset, better calibration
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, accuracy_score,
                              roc_auc_score, confusion_matrix)
import joblib
import warnings
warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════
# 1. LOAD & CLEAN
# ═══════════════════════════════════════════════
df = pd.read_csv("C:/Users/babur/OneDrive/Desktop/ML_floodPrediction/data/processed/hp_landslide_dataset_v7_final.csv")

# Basic cleanup
df['soil_moisture'] = df['soil_moisture'].fillna(df['soil_moisture'].median())
df['ndvi']          = df['ndvi'].clip(-1, 1)
df['aspect']        = df['aspect'].fillna(180)  # default to south-facing

# Drop rows where critical features are missing
df = df.dropna(subset=['rainfall_7day', 'slope', 'elevation', 'label'])

# Drop impossible NDVI values
df = df[df['ndvi'] <= 1.0]
from sklearn.utils import resample

df_majority = df[df['label'] == 0]
df_minority = df[df['label'] == 1]

df_minority_upsampled = resample(
    df_minority,
    replace=True,
    n_samples=len(df_majority),
    random_state=42
)

df = pd.concat([df_majority, df_minority_upsampled])
print("Balanced dataset:", df['label'].value_counts())

print("=== Dataset shape:", df.shape)
print("\nLabel counts:")
print(df['label'].value_counts())
print(f"\nClass balance: {df['label'].mean():.1%} positive")

# ── Correlation check ────────────────────────────────────────────────────
print("\n=== Correlation with label:")
corr = df.corr(numeric_only=True)['label'].sort_values(ascending=False)
print(corr.round(3))

print("\nFeature ranges:")
FEATURE_COLS = ['rainfall_7day', 'slope', 'elevation', 'soil_moisture', 'ndvi', 'aspect']
for col in FEATURE_COLS:
    print(f"  {col:20s}  min={df[col].min():.2f}  max={df[col].max():.2f}  mean={df[col].mean():.2f}")

print("""
IMPORTANT — Unit notes for Flask:
  soil_moisture : absolute units 0-25 (NOT percentage)
  Flask must convert: soil_model = soil_ui_percent / 100 * 25
  aspect        : degrees 0-360 (Flask default = 180, south-facing)
""")

# ═══════════════════════════════════════════════
# 2. FEATURES & SPLIT
# ═══════════════════════════════════════════════
X = df[FEATURE_COLS]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain size: {len(X_train)}, Test size: {len(X_test)}")

# ═══════════════════════════════════════════════
# 3. MODEL WITH CALIBRATION
# ═══════════════════════════════════════════════
base_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', GradientBoostingClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        min_samples_leaf=10,
        subsample=0.8,
        random_state=42
    ))
])

model = CalibratedClassifierCV(base_pipeline, cv=5, method='isotonic')
model.fit(X_train, y_train)

# ═══════════════════════════════════════════════
# 4. EVALUATE
# ═══════════════════════════════════════════════
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
cm  = confusion_matrix(y_test, y_pred)

print("\n" + "="*50)
print(f"Accuracy : {acc:.4f}  ({acc*100:.1f}%)")
print(f"ROC-AUC  : {auc:.4f}")
print(f"\nConfusion Matrix:\n{cm}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Landslide','Landslide']))

cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_acc = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
cv_auc = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
print(f"5-Fold CV Accuracy : {cv_acc.mean():.3f} +/- {cv_acc.std():.3f}")
print(f"5-Fold CV AUC      : {cv_auc.mean():.3f} +/- {cv_auc.std():.3f}")

# ═══════════════════════════════════════════════
# 5. SANITY CHECKS
# ═══════════════════════════════════════════════
THRESHOLD_HIGH   = 0.65
THRESHOLD_MEDIUM = 0.40

def classify(prob):
    if prob >= THRESHOLD_HIGH:   return "HIGH"
    if prob >= THRESHOLD_MEDIUM: return "MEDIUM"
    return "LOW"

def predict_case(rainfall, slope, elevation, soil_moisture=15.0, ndvi=0.45, aspect=180):
    df_in = pd.DataFrame(
        [[rainfall, slope, elevation, soil_moisture, ndvi, aspect]],
        columns=FEATURE_COLS
    )
    prob = model.predict_proba(df_in)[0][1]
    return prob, classify(prob)

print("\n=== Real-world sanity checks ===")
tests = [
    # label,                              rain, slope, elev,  expected
    ("DRY + gentle slope (Shimla)  LOW",     0,   8,  2206,  ["LOW"]),
    ("DRY + steep slope             LOW",    0,  45,  2200,  ["LOW", "MEDIUM"]),
    ("DRY + medium slope            LOW",    0,  25,  1500,  ["LOW", "MEDIUM"]),
    ("Light rain + gentle           LOW",   15,   8,  1200,  ["LOW"]),
    ("Moderate rain + medium      MEDIUM",  60,  28,  2206,  ["MEDIUM", "HIGH"]),
    ("Moderate rain + steep       MEDIUM",  55,  35,  1500,  ["MEDIUM", "HIGH"]),
    ("Shimla heavy rain + steep    HIGH",   80,  45,  2206,  ["HIGH"]),
    ("Manali heavy rain + steep    HIGH",   90,  40,  2050,  ["HIGH"]),
    ("Kullu cloudburst             HIGH",  180,  45,  1220,  ["HIGH"]),
    ("Heavy rain flat terrain       LOW",  150,   5,  1500,  ["LOW", "MEDIUM"]),
]

passed = 0
for label, rain, slp, elev, expected in tests:
    prob, risk = predict_case(rain, slp, elev)
    ok = "PASS" if risk in expected else "FAIL"
    if risk in expected: passed += 1
    print(f"  {ok}  {label}")
    print(f"       rain={rain}mm slope={slp} elev={elev}m -> prob={prob:.3f} {risk}\n")

print(f"Passed: {passed}/{len(tests)}")

# ═══════════════════════════════════════════════
# 6. SAVE
# ═══════════════════════════════════════════════
if passed >= 7:
    joblib.dump(model, "C:/Users/babur/OneDrive/Desktop/ML_floodPrediction/model/landslide_model_v7.pkl")
    print("\nModel saved as landslide_model_v7.pkl")
    print("Update app.py to load landslide_model_v7.pkl")
else:
    print(f"\nOnly {passed}/10 checks passed — review dataset before using.")

Balanced dataset: label
0    475
1    475
Name: count, dtype: int64
=== Dataset shape: (950, 8)

Label counts:
label
0    475
1    475
Name: count, dtype: int64

Class balance: 50.0% positive

=== Correlation with label:
label            1.000
slope            0.321
rainfall_7day    0.294
soil_moisture    0.227
ndvi             0.131
aspect           0.097
elevation       -0.169
Name: label, dtype: float64

Feature ranges:
  rainfall_7day         min=0.00  max=367.06  mean=48.66
  slope                 min=0.00  max=67.22  mean=22.24
  elevation             min=320.00  max=5904.00  mean=2096.76
  soil_moisture         min=1.82  max=25.40  mean=18.19
  ndvi                  min=-1.00  max=1.00  mean=0.37
  aspect                min=-0.00  max=357.51  mean=184.49

IMPORTANT — Unit notes for Flask:
  soil_moisture : absolute units 0-25 (NOT percentage)
  Flask must convert: soil_model = soil_ui_percent / 100 * 25
  aspect        : degrees 0-360 (Flask default = 180, south-facing)


Train 